# myPolar — Traditional ML Baselines
TF-IDF features with Logistic Regression and Linear SVM for all three tasks.

- **Task 1** – binary classification (`polarization`)
- **Task 2 / Task 3** – multi-label classification, one binary classifier per label (`OneVsRestClassifier`)

**Model selection:** hyperparameters are chosen on the **validation** set; the **test** set is used only once, for the final score.  
**Metric:** macro F1.

Run `make-splits.py` first; this notebook reads the files in `splits/`.

In [1]:
import time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, classification_report

SEED = 42
TASKS = {
    "task1": ["polarization"],
    "task2": ["political", "racial/ethnic", "religious", "gender/sexual", "other"],
    "task3": ["stereotype", "vilification", "dehumanization", "extreme_language", "lack_of_empathy", "invalidation"],
}

def load(task):
    d = {s: pd.read_csv(f"splits/{task}_{s}.csv", encoding="utf-8-sig") for s in ["train", "val", "test"]}
    for s in d:
        d[s]["text"] = d[s]["text"].fillna("")
    return d

## Features
The text is already word-segmented, so words are taken as whitespace-separated tokens (`\S+`). The default scikit-learn tokenizer is avoided because it can split Burmese words at vowel signs and other combining marks.

- **word**: word unigrams + bigrams
- **char**: character 2–5-grams inside word boundaries — robust to spelling variation and segmentation errors
- **word+char**: both combined

In [2]:
def make_features(kind):
    word = TfidfVectorizer(token_pattern=r"\S+", ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    char = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True, max_features=200_000)
    return {"word": word, "char": char,
            "word+char": FeatureUnion([("word", word), ("char", char)])}[kind]

def make_model(name, C, multilabel):
    if name == "LogReg":
        m = LogisticRegression(C=C, class_weight="balanced", max_iter=2000, solver="liblinear", random_state=SEED)
    else:
        m = LinearSVC(C=C, class_weight="balanced", max_iter=5000, random_state=SEED)
    return OneVsRestClassifier(m) if multilabel else m

def macro_f1(y_true, y_pred, multilabel):
    # Task 1: macro over the 2 classes (0 and 1). Tasks 2/3: macro over the labels.
    return f1_score(y_true, y_pred, average="macro", zero_division=0)

## Training and model selection

C was selected from a logarithmic grid by validation macro F1; an initial coarse grid was extended until the selected value fell inside the search range for every task.

In [7]:
FEATURES = ["word", "char", "word+char"]
MODELS = ["LogReg", "LinearSVC"]
C_GRID = [0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100]

def run_task(task):
    labels = TASKS[task]
    multilabel = len(labels) > 1
    d = load(task)
    y = {s: d[s][labels].values if multilabel else d[s][labels[0]].values for s in d}
    rows, best = [], {}

    # reference: always predict the most frequent class
    dummy = DummyClassifier(strategy="most_frequent").fit(d["train"]["text"], y["train"])
    rows.append({"features": "-", "model": "Majority", "C": "-",
                 "val_macro_f1": macro_f1(y["val"], dummy.predict(d["val"]["text"]), multilabel),
                 "test_macro_f1": macro_f1(y["test"], dummy.predict(d["test"]["text"]), multilabel)})

    for feat in FEATURES:
        vec = make_features(feat)
        X = {"train": vec.fit_transform(d["train"]["text"])}
        X["val"], X["test"] = vec.transform(d["val"]["text"]), vec.transform(d["test"]["text"])
        for name in MODELS:
            scores = {}
            for C in C_GRID:  # choose C on validation only
                m = make_model(name, C, multilabel).fit(X["train"], y["train"])
                scores[C] = (macro_f1(y["val"], m.predict(X["val"]), multilabel), m)
            C_best = max(scores, key=lambda c: scores[c][0])
            val_f1, model = scores[C_best]
            pred = model.predict(X["test"])
            rows.append({"features": feat, "model": name, "C": C_best,
                         "val_macro_f1": val_f1, "test_macro_f1": macro_f1(y["test"], pred, multilabel)})
            if val_f1 > best.get("val_f1", -1):
                best = {"val_f1": val_f1, "name": f"{name} ({feat}, C={C_best})", "pred": pred, "y": y["test"]}

    res = pd.DataFrame(rows).round(4)
    res.insert(0, "task", task)
    return res, best, labels

results, bests = [], {}
for task in TASKS:
    t = time.time()
    res, best, labels = run_task(task)
    results.append(res); bests[task] = (best, labels)
    print(f"{task} done in {time.time() - t:.0f}s")

task1 done in 82s


c:\Users\ASUS\miniconda3\envs\hostelbot\lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\ASUS\miniconda3\envs\hostelbot\lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\ASUS\miniconda3\envs\hostelbot\lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\ASUS\miniconda3\envs\hostelbot\lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\ASUS\miniconda3\envs\hostelbot\lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\ASUS\miniconda3\envs\hostelbot\lib\site-packages\sklearn

task2 done in 316s


c:\Users\ASUS\miniconda3\envs\hostelbot\lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\ASUS\miniconda3\envs\hostelbot\lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\ASUS\miniconda3\envs\hostelbot\lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\ASUS\miniconda3\envs\hostelbot\lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\ASUS\miniconda3\envs\hostelbot\lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\ASUS\miniconda3\envs\hostelbot\lib\site-packages\sklearn

task3 done in 568s


## Results
The best model per task is the one with the highest **validation** macro F1 (not test).

In [8]:
all_res = pd.concat(results, ignore_index=True)
for task in TASKS:
    print(f"\n=== {task} ===")
    print(all_res[all_res.task == task].drop(columns="task").sort_values("val_macro_f1", ascending=False).to_string(index=False))


=== task1 ===
 features     model    C  val_macro_f1  test_macro_f1
word+char    LogReg    3        0.7964         0.7899
     char    LogReg    3        0.7917         0.7823
word+char LinearSVC  0.3        0.7898         0.7917
     char LinearSVC  0.3        0.7889         0.7833
     word LinearSVC  0.3        0.7394         0.7478
     word    LogReg    3        0.7376         0.7506
        -  Majority    -        0.3389         0.3389

=== task2 ===
 features     model     C  val_macro_f1  test_macro_f1
     char LinearSVC   0.1        0.5640         0.5716
     char    LogReg     1        0.5634         0.5733
word+char LinearSVC  0.03        0.5605         0.5470
word+char    LogReg   0.3        0.5585         0.5457
     word LinearSVC   0.1        0.4095         0.4101
     word    LogReg     1        0.4064         0.4222
        -  Majority     -        0.0000         0.0000

=== task3 ===
 features     model    C  val_macro_f1  test_macro_f1
     char LinearSVC  0.3     

## Per-label report for the best model of each task

In [9]:
for task, (best, labels) in bests.items():
    print(f"\n=== {task}: {best['name']} ===")
    names = ["not polarized", "polarized"] if len(labels) == 1 else labels
    print(classification_report(best["y"], best["pred"], target_names=names, digits=3, zero_division=0))


=== task1: LogReg (word+char, C=3) ===
               precision    recall  f1-score   support

not polarized      0.780     0.793     0.786       522
    polarized      0.800     0.787     0.793       549

     accuracy                          0.790      1071
    macro avg      0.790     0.790     0.790      1071
 weighted avg      0.790     0.790     0.790      1071


=== task2: LinearSVC (char, C=0.1) ===
               precision    recall  f1-score   support

    political      0.635     0.762     0.693       265
racial/ethnic      0.522     0.659     0.583        91
    religious      0.457     0.538     0.494        39
gender/sexual      0.516     0.723     0.603        65
        other      0.388     0.650     0.486       157

    micro avg      0.519     0.700     0.596       617
    macro avg      0.504     0.667     0.572       617
 weighted avg      0.532     0.700     0.602       617
  samples avg      0.305     0.356     0.318       617


=== task3: LinearSVC (char, C=0.3

## Summary

In [10]:
summary = (all_res[all_res.model != "Majority"]
           .sort_values("val_macro_f1", ascending=False).groupby("task").head(1)
           [["task", "model", "features", "C", "val_macro_f1", "test_macro_f1"]])
majority = all_res[all_res.model == "Majority"][["task", "test_macro_f1"]].rename(columns={"test_macro_f1": "majority_test_f1"})
summary = summary.merge(majority, on="task")
all_res.to_csv("baseline_results.csv", index=False)
summary

,task,model,features,C,val_macro_f1,test_macro_f1,majority_test_f1
0,task1,LogReg,word+char,3,0.7964,0.7899,0.3389
1,task2,LinearSVC,char,0.1,0.5640,0.5716,0.0000
2,task3,LinearSVC,char,0.3,0.5242,0.5092,0.0000


**Traditional ML baseline.** TF-IDF with a linear classifier reached a macro F1 of 0.790 on Task 1, 0.572 on Task 2 and 0.509 on Task 3. Validation and test scores differ by less than 0.01 for every task, indicating stable model selection.

Per-label results show that performance depends more on how lexically distinctive a label is than on its frequency. In Task 2, *gender/sexual* (65 test examples) outperformed *racial/ethnic* (91), while the heterogeneous *other* category scored low (0.486) despite being the second most frequent. In Task 3, *vilification*, which is typically expressed through explicit insults, scored highest (0.646), while labels that depend on meaning and tone rather than specific words — *stereotype*, *dehumanization*, *lack of empathy* and *invalidation* — all scored between 0.45 and 0.48. For Tasks 2 and 3, recall exceeded precision for every label, reflecting the balanced class weighting.